# Drivers Graphs

This notebook is made to fetch driver telemetry data in a single track in a single year

In [14]:
import fastf1
import os

YEAR = 2024
DRIVER = "max_verstappen"
RACE = 3

session = fastf1.get_session(YEAR, RACE, "R")
session.load(laps=True, telemetry=True, weather=False)
driver_slug = f"{DRIVER}_{YEAR}"
for _, row in session.results.iterrows():
    if row["DriverId"] == DRIVER:
        driver_abv = row["Abbreviation"]
        break
driver_laps = session.laps.pick_drivers(driver_abv)
fastest_lap = driver_laps.pick_fastest()
start_td = fastest_lap["Time"] - fastest_lap["LapTime"]
end_td = fastest_lap["Time"]
telemetry = fastest_lap.get_telemetry()
telemetry['RelativeSeconds'] = (telemetry['SessionTime'] - start_td).dt.total_seconds()

position_points = []
for row in telemetry.itertuples():
    position_points.append((
        round(row.RelativeSeconds, 4), 
        row.X, 
        row.Y,
        row.Z,
        row.RPM, 
        row.Speed, 
        row.nGear, 
        row.Throttle, 
        row.Brake, 
        row.DRS, 
        row.Status
    ))

os.makedirs('data', exist_ok=True)
filename = f'data/telemetry_{DRIVER}_{YEAR}_{RACE}.csv'

with open(filename, 'w') as f:
    f.write("seconds,x,y,z,rpm,speed,gear,throttle,brake,drs,status\n")
    for point in position_points:
        f.write(','.join(map(str, point)) + '\n')

print(f"Exported {len(position_points)} points for {DRIVER} in {RACE}")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data fo

Exported 652 points for max_verstappen in 3
